# Weekly Challenge 01 – Second Opinion 🩺
*Tabular classification · difficulty ★☆☆☆ · ~1 evening*

A hospital wants a **second-opinion tool** for breast-mass biopsies. For every sample, a lab has measured 30 numbers
from a digitised image of the cell nuclei (radius, texture, smoothness, …). Your model must say **malignant or benign**.

A colleague already built a k-nearest-neighbours classifier. It gets about 92 % test accuracy – that is
roughly 13 wrong diagnoses out of 171. Your job: **reach at least 97 % test accuracy** (at most 5 mistakes).

| | |
|---|---|
| **Data** | `sklearn.datasets.load_breast_cancer` – 569 samples, 30 numeric features (bundled, works offline) |
| **Split** | stratified 70/30, `random_state=42` (fixed – don't change it) |
| **Metric** | accuracy on the test set |
| **Baseline** | k-NN (k = 5) on raw features → **0.924** |
| **Target** | **≥ 0.97** |

Theory: [[k-Nearest Neighbors]] · [[Data Preprocessing]] · [[Logistic Regression]] · [[Cross-Validation and Model Selection]]

**How it works**
1. Read the story, run the setup and the baseline.
2. Build your model in the ✏️ *Your turn* cell (it starts as a copy of the baseline).
3. Run the 🏁 *Evaluate* cell – it prints ✅ if you beat the target, ❌ otherwise.
4. Stuck? Open the 💡 hints one at a time. Done (or truly stuck)? Open the 🏆 solution walkthrough.
5. Log your best score in [[Weekly Challenges]].

**Rules** – make every modelling decision with the training data only (cross-validation or a validation split).
The test set is touched exactly once, by the evaluate cell. Peeking at it to tune is cheating yourself.

In [ ]:
import numpy as np, pandas as pd, warnings
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
warnings.filterwarnings("ignore")

def report(name, score, baseline, target, higher_is_better=True):
    beat = score >= target if higher_is_better else score <= target
    arrow = "≥" if higher_is_better else "≤"
    print(f"{name}: {score:.4f}   (baseline {baseline:.4f}, target {arrow} {target})")
    print(("✅ Target beaten – log it in Weekly Challenges!" if beat
           else "❌ Not yet – keep going (or open a hint)."))

## 1 · Load the data (fixed split)

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target            # target: 0 = malignant, 1 = benign
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print("class balance (train):", y_train.value_counts(normalize=True).round(3).to_dict())

## 2 · Have a look
Look at the *scale* of the columns. Anything that would bother a distance-based model?

In [ ]:
X_train.describe().T[["mean", "std", "min", "max"]].sort_values("std", ascending=False).head(8)

## 3 · Baseline

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

baseline = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
BASELINE = accuracy_score(y_test, baseline.predict(X_test))
TARGET = 0.97
print(f"baseline test accuracy: {BASELINE:.3f}")

## 4 · ✏️ Your turn
Define `my_model` – any scikit-learn estimator or pipeline. Use cross-validation on the **training** set to compare ideas.

In [ ]:
# ✏️ TODO: replace this with your own model
my_model = KNeighborsClassifier(n_neighbors=5)

cv = cross_val_score(my_model, X_train, y_train, cv=5, scoring="accuracy")
print(f"5-fold CV accuracy on train: {cv.mean():.3f} ± {cv.std():.3f}")

## 5 · 🏁 Evaluate (run once you're happy with your CV score)

In [ ]:
my_model.fit(X_train, y_train)
score = accuracy_score(y_test, my_model.predict(X_test))
report("test accuracy", score, BASELINE, TARGET)

<details><summary>💡 Hint 1</summary>

The table in step 2 shows `worst area` in the thousands and `smoothness` around 0.1. What does a Euclidean distance mostly measure then?
</details>

<details><summary>💡 Hint 2</summary>

Put a `StandardScaler` in front of the model with `make_pipeline` (inside the pipeline, so CV stays leak-free).
</details>

<details><summary>💡 Hint 3</summary>

Scaling alone takes k-NN to about 0.96. Try a linear model, too – the classes are almost linearly separable.
</details>

<details>
<summary>🏆 <b>Solution walkthrough</b> (click to open – try first!)</summary>

**Why the baseline is weak.** k-NN decides by Euclidean distance. Unscaled, `worst area` (values up to ~4000) dominates
the distance and the other 29 features barely count – the model is effectively one-dimensional.

**Step 1 – scale.** `StandardScaler` inside a pipeline gives every feature unit variance. k-NN jumps to ≈ 0.959 test
accuracy – better, but still short of 0.97.

**Step 2 – a better model family.** With 30 informative, roughly linearly separable features and only ~400 training
samples, a regularised linear model is a great fit. Scaled logistic regression scores ≈ 0.98 in 5-fold CV and
**0.988 on the test set (2 mistakes out of 171)**. An RBF-SVM on scaled features also clears the bar (≈ 0.977).

**Lesson.** Always ask "what does my model's notion of similarity/size depend on?" – and try a simple linear model before anything fancy.

```python
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

my_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
cv = cross_val_score(my_model, X_train, y_train, cv=5, scoring="accuracy")
print(f"5-fold CV accuracy on train: {cv.mean():.3f} ± {cv.std():.3f}")
```
</details>